In [ ]:
from pathlib import Path

import geopandas as gpd
import rasterio
from rasterio.mask import mask

# Input wind raster (unclipped)
WIND_RASTER = Path(r"C:\HyGalapagos\inputs\wind-speed_100m.tif")

# Try common boundary locations. Update this list if your boundary is elsewhere.
BOUNDARY_CANDIDATES = [
    Path(r"C:\Galapagos_IERSE\01_Organizacion_Territorial\Provincia_Galapagos_CONALI_2018.shp"),
    Path(r"C:\HyGalapagos\data\Parroquias_Galapagos_CONALI_2018.shp"),
    Path(r"C:\HyGalapagos\data\galapagos_boundary.shp"),
    Path(r"C:\HyGalapagos\data\galapagos_boundary.gpkg"),
]

# Output clipped raster
OUTPUT_RASTER = Path(r"C:\HyGalapagos\inputs\wind-speed_100m_galapagos_clipped.tif")
OUTPUT_RASTER.parent.mkdir(parents=True, exist_ok=True)


def first_existing_path(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError(
        "No Galapagos boundary file found. Checked:\n- " + "\n- ".join(str(p) for p in paths)
    )


boundary_path = first_existing_path(BOUNDARY_CANDIDATES)
boundary = gpd.read_file(boundary_path)

with rasterio.open(WIND_RASTER) as src:
    if boundary.crs != src.crs:
        boundary = boundary.to_crs(src.crs)

    geoms = [geom for geom in boundary.geometry if geom is not None and not geom.is_empty]
    if not geoms:
        raise ValueError("Boundary dataset has no valid geometries.")

    nodata = src.nodata if src.nodata is not None else -9999.0
    clipped_data, clipped_transform = mask(
        src,
        geoms,
        crop=True,
        filled=True,
        nodata=nodata,
    )

    out_meta = src.meta.copy()
    out_meta.update(
        {
            "driver": "GTiff",
            "height": clipped_data.shape[1],
            "width": clipped_data.shape[2],
            "transform": clipped_transform,
            "nodata": nodata,
        }
    )

    with rasterio.open(OUTPUT_RASTER, "w", **out_meta) as dst:
        dst.write(clipped_data)

print("Boundary used:", boundary_path)
print("Saved clipped raster:", OUTPUT_RASTER)
print("Outside boundary values set to NoData:", nodata)


Boundary used: C:\Galapagos_IERSE\01_Organizacion_Territorial\Provincia_Galapagos_CONALI_2018.shp
Saved clipped raster: C:\HyGalapagos\outputs_eolico\wind-speed_100m_galapagos_clipped.tif
Outside boundary values set to NoData: nan
